# 11 — Local Phi-3 versus hosted Gemini

Iteration 10 uses the installed Phi-3 Mini 3.8B Q4_0 model through Ollama. Both models perform the same clause query-expansion task with the same prompt, output schema, temperature, token cap, validators, and stemmed BM25 reference. Original query routing is retained locally. No gold answers or test questions enter model inputs.

All outputs are cached. Running this notebook contacts neither Gemini nor Ollama. It reviews measured train results and runs saved rewrite examples. Phi-3's six invalid outputs use the original BM25 query; those questions remain in the denominator.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
if ROOT.name=='notebooks':
    ROOT=ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
from improve import read_log,read_json
from search_service import build_model, search, EXAMPLE_QUERY

runs={r['iteration']:r for r in read_log(ROOT/'experiments/llm_log.jsonl')}
assert 9 in runs and 10 in runs
rows=[]
for label,metrics in [('original baseline',runs[10]['metrics']['baseline']),
                       ('best non-LLM',runs[10]['metrics']['reference']),
                       ('Gemini',runs[9]['metrics']['candidate']),
                       ('Phi-3',runs[10]['metrics']['candidate'])]:
    m=next(m for m in metrics['retrieval'] if m['k']==5)
    rows.append(dict(approach=label,precision=m['precision'],recall=m['recall'],
                     benchmark_precision=m['benchmark_precision'],benchmark_recall=m['benchmark_recall'],
                     false_abstention=metrics['false_abstention'],correct_abstention=metrics['correct_abstention'],
                     replay_p95_ms=metrics['latency_ms']['p95']))
display(pd.DataFrame(rows))

,approach,precision,recall,benchmark_precision,benchmark_recall,false_abstention,correct_abstention,replay_p95_ms
0,original baseline,0.017433,0.181647,0.017594,0.192427,0.0,0.0,8.619305
1,best non-LLM,0.078798,0.670075,0.083766,0.765975,0.0,0.0,10.308020
2,Gemini,0.084691,0.790986,0.089255,0.895841,0.0,0.0,15.954550
3,Phi-3,0.080682,0.707500,0.085128,0.800671,0.0,0.0,15.204055


## Compare quality, reliability, and latency

Operational acceptance uses the original first-use latency guardrail. Replay-mode acceptance is recorded separately; fast replay must not hide generation time. Estimates combine separately measured model calls and retrieval, so they are not production load tests.

The first local call included model loading: about 52 seconds total, including 37 seconds of loading. Other calls used the resident model. The runtime snapshot records approximately 3 GB VRAM allocation on a machine with about 16 GB RAM. Hardware, quantization, tokenizers, and provider implementations differ; this is a controlled application comparison, not an equal-compute model benchmark.

In [3]:
for i,label in [(9,'Gemini'),(10,'Phi-3')]:
    r=runs[i]
    print(label)
    print('Model/config:',r['model_config'])
    print('Output validation:',r['cache_status_counts'])
    print('First-use estimated latency (ms):',r['estimated_first_use_latency_ms'])
    print('Generation total seconds:',r['generation_total_ms']/1000)
    print('Usage:',r['aggregate_unique_request_usage'])
    print('Cost note:',r['billing_note'])
    print('Operational:',r['decision'])
    print('Replay:',r['replay_decision'])

Gemini
Model/config: {'name': 'llm_gemini_rewrite', 'provider': 'gemini', 'model': 'gemini-3.5-flash-lite', 'model_revision': 'stable identifier; actual modelVersion recorded in every response', 'runtime': 'gemini-rest-v1beta', 'prompt_file': 'prompts/retrieval_rewrite_v1.txt', 'temperature': 0, 'max_output_tokens': 256, 'thinking_level': 'minimal', 'baseline_config': 'configs/bm25_focused_stemmed.json'}
Output validation: {'valid': 53}
First-use estimated latency (ms): {'median': 1458.874149946496, 'p95': 2578.9265950152185}
Generation total seconds: 90.64178400009405
Usage: {'promptTokenCount': 8257, 'candidatesTokenCount': 1284, 'totalTokenCount': 9541}
Cost note: Token usage recorded; actual charge is not inferred without account billing data.
Operational: {'accepted': False, 'checks': {'recall_gain': True, 'precision': True, 'answerable_abstention': True, 'unanswerable_abstention': True, 'latency': False}}
Replay: {'accepted': True, 'checks': {'recall_gain': True, 'precision': Tru

## Try either model without model services

Select `llm_gemini_rewrite`, `llm_local_phi3_rewrite`, or `bm25_focused_stemmed`. The three documented presentation questions are cached for both models. Uncached wording fails clearly; there is no live-generation fallback in the demo.

In [4]:
configuration='llm_local_phi3_rewrite'
model=build_model(configuration,ROOT)
result=search(model,EXAMPLE_QUERY,k=5)
print(result['configuration'],result['status'])
if result['hits']:
    display(pd.DataFrame(result['hits'])[['score','file_path','start','end','text']])
else:
    print(result['message'])

llm_local_phi3_rewrite matches


,score,file_path,start,end,text
0,13.208989,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,7.683611,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


## Inspect invalid local outputs

Validation failures are retained, not regenerated until they pass. This helps explain the reliability tradeoff of a small local model. Original BM25 search is used for these inputs.

In [5]:
from llm_rewrite import get_rewrite,validate_output
from prepare_llm import train_questions
config=read_json(ROOT/'llm_configs/local_phi3.json')
invalid=[]
for q in train_questions(ROOT):
    entry,_=get_rewrite(ROOT,config,q,cache_only=True)
    if entry['status']!='valid':
        try:
            validate_output(entry['raw_text'],q)
            reason='Provider failure'
        except (ValueError,TypeError) as error:
            reason=str(error)
        invalid.append(dict(question=q,status=entry['status'],reason=reason,raw_output=entry['raw_text']))
display(pd.DataFrame(invalid))

,question,status,reason,raw_output
0,Are there any services to be provided after th...,invalid_output,Unterminated string starting at: line 2 column...,"{\n ""search_terms"": ""post-termination obligat..."
1,Is there a minimum commitment required under t...,invalid_output,Unterminated string starting at: line 2 column...,"{\n ""search_terms"": ""minimum commitment claus..."
2,What are the audit rights under this contract?,invalid_output,Unterminated string starting at: line 2 column...,"{\n ""search_terms"": ""audit rights, contract r..."
3,What licenses are granted under this contract?,invalid_output,Unterminated string starting at: line 2 column...,"{\n ""search_terms"": ""license provisions, gran..."
4,What notice is required to terminate the agree...,invalid_output,Invalid search term length.,"{\n ""search_terms"": ""termination notice, cont..."
5,Which clause requires multi-factor authenticat...,invalid_output,Invalid search term length.,"{\n ""search_terms"": ""multi-factor authenticat..."


## Explain the result

“We tested hosted and local models on the same query-expansion task. Both improved BM25 train recall, but Gemini reached 79.10% compared with Phi-3's 70.75% and the non-LLM reference's 67.01%. Phi-3 avoids external inference transfer and hosted model fees, but uses local resources and produced six invalid outputs. Caches make both presentation paths reproducible without model calls.”

Both models still return passages for every train negative. Neither passes the original first-use latency rule, so the best accepted non-LLM system remains the default. Median local first-use estimate is lower than Gemini's, but local p95 is higher; do not describe either as universally faster.

All ten distinct improvement iterations are complete. No test performance has been measured. Remaining required work is final configuration freeze, held-out evaluation with real run counting, final failure analysis, report, and reproduction checks. Do not start more trials or the deferred FAISS/hybrid work as part of this checkpoint.